# FinChain OPD / Online-DPO RunPod Notebook

OPD here means current-policy teacher signal, not fixed offline DPO. The repo supports two paths:

1. Bridge OPD: sample current-policy rollouts, verify them, convert best-vs-worst outputs into DPO-compatible pairs, train, then repeat.
2. Industry online OPD: use TRL OnlineDPOTrainer so generation, scoring, and updates happen inside the trainer loop.

Use the bridge when you want maximum inspectability. Use TRL OnlineDPO when you want the behavior industry code is converging toward.

## RunPod Terminal Preflight
Run this in a JupyterLab terminal before opening the notebook cells:

```bash
cd /workspace
if [ ! -d finpost ]; then
  git clone https://github.com/shannan-liu1/finpost.git
fi
cd /workspace/finpost
git pull --ff-only
pip install -e ".[dev,rlvr]"
python -c "import finpost, torch; print('finpost ok'); print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"

export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain/test.jsonl
export WANDB_MODE=offline
```

If `import finpost` fails after `pip install -e`, use the fallback from `docs/runbooks/runpod-bootstrap.md`:

```bash
echo "/workspace/finpost/src" > /usr/local/lib/python3.11/dist-packages/finpost.pth
python -c "import finpost; print(finpost.__file__)"
```

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO = Path('/workspace/finpost') if Path('/workspace/finpost').exists() else Path.cwd()
os.chdir(REPO)

MODEL_ID = 'Qwen/Qwen2.5-1.5B'
SFT_CONFIG = 'experiments/finchain/qwen25_1_5b_sft_runpod.yaml'
SFT_RUN_ROOT = Path('results/checkpoints/qwen25-1p5b-finchain-sft')
SFT_HF_DIR = Path('results/checkpoints/qwen25-1p5b-finchain-sft-hf')
DPO_CONFIG = 'experiments/dpo/finchain_qwen25_1_5b.yaml'
PAIR_RUN = Path('results/finchain_pairs/run_001')

os.environ.setdefault('WANDB_MODE', 'offline')


def run(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=check, text=True)

print('repo:', REPO)
print('WANDB_MODE:', os.environ.get('WANDB_MODE'))

In [ ]:
import torch

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 1), 'GB')
else:
    print('No CUDA GPU visible. Stop here on RunPod.')

In [ ]:
from finpost.data.finchain import load_finchain, resolve_finchain_path

for split in ['train', 'validation', 'test']:
    path = resolve_finchain_path(split)
    print(split, '->', path, 'exists=', path.exists())

train_examples = load_finchain('train')
print('train examples:', len(train_examples))
print('first prompt id:', train_examples[0].id)
print('first gold answer:', train_examples[0].final_answer)

## Bridge OPD: One Cheap Round
This is the transparent version. It reuses `scripts/build_dpo_pairs.py` but the policy checkpoint is whatever model you are currently improving. After training, you can point `--model-checkpoint` at the new checkpoint and repeat.

This is OPD-like because the data distribution refreshes from the current policy. It is not the same as true online-DPO because generation and training are still separate phases.

In [ ]:
bridge_round_cmd = f"""
CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py \
  --model-checkpoint {SFT_HF_DIR} \
  --sources finchain \
  --out-dir results/finchain_opd/round_001/pairs \
  --heldout-train-n 512 \
  --samples-per-prompt 6 \
  --generation-batch-size 32 \
  --max-new-tokens-finchain 512 \
  --max-pairs-per-prompt 4

WANDB_MODE=offline python -m finpost.training.dpo_train \
  --config experiments/dpo/finchain_qwen25_1_5b.yaml \
  --device cuda \
  --max-steps 100
""".strip()
print(bridge_round_cmd)

## Two-GPU Bridge OPD Rollouts
Use this when rollouts dominate cost. Training can remain single GPU, because pair construction is usually the bottleneck for small 1.5B runs.

```bash
cd /workspace/finpost
export CKPT=results/checkpoints/qwen25-1p5b-finchain-sft-hf
export OUT=results/finchain_opd/round_001

CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-00-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 0 --num-shards 2 &
CUDA_VISIBLE_DEVICES=1 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-01-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 1 --num-shards 2 &
wait
python scripts/merge_dpo_pair_shards.py --shard-dirs $OUT/shards/shard-00-of-02 $OUT/shards/shard-01-of-02 --out-dir $OUT/merged
```

## Industry Online-DPO / OPD Path
This path uses TRL's experimental OnlineDPOTrainer. It only needs prompt rows; the trainer samples current-policy responses and calls the FinChain verifier reward function. That is closer to your teacher/student mental model than offline DPO.

In [ ]:
prompt_export_cmd = "python scripts/export_finchain_prompts.py --split train --n 2000 --out-path results/finchain_rlvr/prompts_train_2000.jsonl"
print(prompt_export_cmd)
print('
Single GPU TRL OnlineDPO canary:')
print('python scripts/train_finchain_trl_online_dpo.py --model', MODEL_ID, '--train-n 256 --max-steps 20 --output-dir results/checkpoints/qwen25-1p5b-finchain-online-dpo-canary')
print('
Two GPU TRL OnlineDPO:')
print('accelerate launch --num_processes 2 scripts/train_finchain_trl_online_dpo.py --model', MODEL_ID, '--train-n 2000 --max-steps 300 --output-dir results/checkpoints/qwen25-1p5b-finchain-online-dpo-2gpu')

## What To Watch
Track `objective/kl`, reward margin, parse success, and held-out FinChain accuracy. If reward goes up while final-answer accuracy is flat, you are probably learning answer-format shortcuts. If KL spikes early, cut the learning rate or increase beta. If almost every prompt is all-correct or all-wrong, increase samples per prompt or return to SFT before spending on OPD.